# O‘zMSt 285:2024, 1-algoritm: 2- va 3-tavsiyalarni amalga oshirish va tekshirish

**2-tavsiya.** Aralash va TuzilmaKalit: diamatritsa 2 moduli bo‘yicha har doim teskarilanuvchi bo‘lishi uchun
raund kaliti kₑ ning barcha elementlari toq qilinadi (kₑ ← kₑ ∨ 1).

**3-tavsiya.** Raund funksiyasiga yuqori bitlarni quyi bitlarga o‘tkazuvchi element qo‘shiladi
(SurHolat dan keyin, ikkinchi Qo‘sh dan oldin): davriy aylantirish, XOR-surish yoki S-box qatlami.

Daftar mustaqil: kerakli modullar 0-hujayrada yuklanadi. Faqat `numpy` kerak.
`TEZ = True` qilinsa, tanlanmalar kichraytiriladi (tez sinov uchun).

In [1]:
# 0-hujayra. Dastur modullarini yuklash (daftar mustaqil ishlaydi).
# Agar modullar shu papkada bo'lsa, ular ishlatiladi; bo'lmasa quyidagi matndan xotirada yaratiladi.
import sys, types, importlib.util
_MODULLAR = {
    'algoritm': '"""\nO‘zMSt 285:2024, 1-algoritm: vektorlashtirilgan realizatsiya.\nVariantlar: p = 256 yoki p = 16; yutish o‘zgartirishi bz = \'std\' (BaytZichlash) yoki \'sbz\' (S(h ⊕ S(n))).\nBarcha funksiyalar (N, 32) o‘lchamli numpy massivlar ustida ishlaydi: bir vaqtda N ta mustaqil hisob.\n"""\nimport numpy as np\n\n# B ilova (majburiy): 1x256 chiziqli zichlash massivi (satr — yuqori yarim bayt)\nZICHLASH = bytes.fromhex(\n    "050d060b010a0f0800040709020c030e"\n    "0807020e0f030b06010c0d0a05040900"\n    "0e020d040c07010b06090005030a080f"\n    "000e090c030d07040f0605010b020a08"\n    "030a0702040c09010e0d0f0800050b06"\n    "02030108000e05090c0b06070a0f0d04"\n    "0a040e0f090508020b0001030c06070d"\n    "0b090a0106040d0f03050e000807020c"\n    "010003070d0b0a0c090e04060f080502"\n    "04080b090e0602050a030c0f070d0001"\n    "090c0f0002010e0a05080b0d04030607"\n    "060b080d07090003040f0a020e010c05"\n    "0f0100050a0803070d02090c060e040b"\n    "0c05040a0b02060d0807030e01000f09"\n    "070f0c060500040e020a080b0d090103"\n    "0d060503080f0c0007010204090b0e0a")\nZ = np.array(list(ZICHLASH), dtype=np.int64)\ndef aes_sbox():\n    s=[0]*256; p=q=1\n    while True:\n        p ^= (p<<1) ^ (0x1b if p&0x80 else 0); p&=0xff\n        q ^= q<<1; q ^= q<<2; q ^= q<<4; q&=0xff\n        if q&0x80: q^=0x09\n        x=q ^ ((q<<1|q>>7)&0xff) ^ ((q<<2|q>>6)&0xff) ^ ((q<<3|q>>5)&0xff) ^ ((q<<4|q>>4)&0xff)\n        s[p]=x^0x63\n        if p==1: break\n    s[0]=0x63; return np.array(s,dtype=np.int64)\nS=aes_sbox()\nclass Cfg:\n    def __init__(s,p=256,bz=\'sbz\'):\n        s.p=p; s.bz=bz\n        s.INV=np.zeros(p,dtype=np.int64)\n        for x in range(1,p,2): s.INV[x]=pow(x,-1,p)\ndef prododd(a,p):\n    b=np.where(a%2==0,a+1,a)%p; r=np.ones(a.shape[0],dtype=np.int64)\n    for i in range(a.shape[1]): r=(r*b[:,i])%p\n    return r\ndef dK(P,Jo,p): return np.where((P==(Jo+1)%p)|(P==(-(Jo+1))%p),10,0)\ndef step(x,A,B,R,c):\n    d=(B+R*x)%c.p; return (-A*x*c.INV[d])%c.p\ndef qosh(h,hn,R,c):\n    p=c.p; R=R[:,None]\n    J1h=h.sum(1)%p; J1hn=hn.sum(1)%p\n    Ph=prododd(h,p); Phn=prododd(hn,p)\n    K1h=(Ph+dK(Ph,J1hn,p))%p; K1hn=(Phn+dK(Phn,J1h,p))%p\n    dAh=np.where(J1hn%2==1,0,1); dAhn=np.where(J1h%2==1,0,1)\n    A1h=(J1hn[:,None]+2*hn+dAh[:,None])%p; A1hn=(J1h[:,None]+2*h+dAhn[:,None])%p\n    h1=step(h,A1h,(K1h[:,None]-2*h)%p,R,c); hn1=step(hn,A1hn,(K1hn[:,None]-2*hn)%p,R,c)\n    J2h=h1.sum(1)%p; J2hn=hn1.sum(1)%p\n    P2h=prododd(h1,p); P2hn=prododd(hn1,p)\n    K2h=(P2h+dK(P2h,J2hn,p))%p; K2hn=(P2hn+dK(P2hn,J2h,p))%p\n    dA2h=np.where((J1hn+J2hn)%2==1,0,1); dA2hn=np.where((J1h+J2h)%2==1,0,1)\n    A2h=(J1hn[:,None]+J2hn[:,None]+2*hn1+dA2h[:,None])%p; A2h=np.where(A2h==0,1,A2h)\n    A2hn=(J1h[:,None]+J2h[:,None]+2*h1+dA2hn[:,None])%p; A2hn=np.where(A2hn==0,1,A2hn)\n    B2h=((K1h*K2h)[:,None]-2*h1)%p; B2hn=((K1hn*K2hn)[:,None]-2*hn1)%p\n    h2=step(h1,A2h,B2h,R,c); h2=np.where(h2==0,(A2h+B2h)%p,h2)\n    hn2=step(hn1,A2hn,B2hn,R,c); hn2=np.where(hn2==0,(A2hn+B2hn)%p,hn2)\n    return step(h2,A2h,B2h,R,c), step(hn2,A2hn,B2hn,R,c)\ndef bzf(h,hn,c):\n    if c.p==256:\n        if c.bz==\'std\': return (Z[h]<<4)|Z[hn]\n        return S[h^S[hn]]\n    if c.bz==\'std\': return h^hn\n    H=16*h[:,0::2]+h[:,1::2]; N=16*hn[:,0::2]+hn[:,1::2]; Q=S[H^S[N]]\n    out=np.empty_like(h); out[:,0::2]=Q>>4; out[:,1::2]=Q&15; return out\ndef dia(H,K,p):  # H,K: (N,4,4)\n    sk=K.sum(1)            # sum_i K[i,u]  -> (N,4)\n    sh=H.sum(1)            # sum_i H[i,u]\n    R=np.empty_like(H)\n    for s in range(4):\n        for u in range(4):\n            if s==u:\n                v=H[:,u,u]*sk[:,u]-sum(H[:,i,i]*K[:,i,u] for i in range(4) if i!=u)\n            else:\n                v=H[:,s,u]*sk[:,u]+K[:,s,u]*sh[:,u]-sum(H[:,s,i]*K[:,i,u] for i in range(4) if i not in (s,u))\n            R[:,s,u]=v%p\n    return R\ndef aralash(h,ke,c):\n    H=h.reshape(-1,4,8); K=ke.reshape(-1,4,8)\n    L=dia(H[:,:,:4],K[:,:,:4],c.p); Rr=dia(H[:,:,4:],K[:,:,4:],c.p)\n    return np.concatenate([L,Rr],axis=2).reshape(-1,32)\ndef _shift(h,rows,cols):\n    m=h.reshape(-1,4,8).copy()\n    for s in range(4): m[:,s,:]=np.roll(m[:,s,:],rows[s],axis=1)\n    for u,n in cols.items(): m[:,:,u]=np.roll(m[:,:,u],n,axis=1)\n    return m.reshape(-1,32)\ndef surholat(h): return _shift(h,[1,2,3,4],{0:1,4:1,1:2,5:2,2:3,6:3})\ndef surkalit(ke): return _shift(ke,[4,3,2,1],{1:-3,5:-3,2:-2,6:-2,3:-1,7:-1})\ndef tuz(ke,k,c):\n    p=c.p\n    return np.where((ke==0)&(k%2==0),2,np.where((ke!=0)&(ke%2==k%2),ke,(ke-1)%p))\ndef iteration(holat,blok,ke,k,c,trace=None,rounds=10):\n    """Bitta tashqi iteratsiya (i): Qo\'sh+BZ, so\'ng \'rounds\' ta bosqich."""\n    R=k[:,0]\n    def T(name,j,h,kk):\n        if trace is not None: trace.append((name,j,h.copy(),kk.copy()))\n    holat,hn=qosh(holat,blok,R,c); T(\'Qosh0\',0,holat,ke)\n    holat=bzf(holat,hn,c); T(\'BZ0\',0,holat,ke)\n    for j in range(1,rounds+1):\n        holat=aralash(holat,ke,c); T(\'Aralash\',j,holat,ke)\n        holat,ke=qosh(holat,ke,R,c); T(\'Qosh1\',j,holat,ke)\n        holat=surholat(holat); ke=surkalit(ke); T(\'Sur\',j,holat,ke)\n        if j==8:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj8\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ8\',j,holat,ke)\n        else:\n            holat,ke=qosh(holat,ke,R,c); T(\'Qosh2\',j,holat,ke)\n        ke=tuz(ke,k,c); T(\'Tuz\',j,holat,ke)\n        if j==9:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj9\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ9\',j,holat,ke)\n    return holat,ke\ndef to_units(b,p):\n    b=np.asarray(b,dtype=np.int64)\n    if p==256: return b\n    out=np.empty(b.shape[:-1]+(b.shape[-1]*2,),dtype=np.int64); out[...,0::2]=b>>4; out[...,1::2]=b&15; return out\ndef hash_msgs(msgs,keys,c):\n    """msgs: (N,L) bayt, L - blok uzunligiga karrali; keys: (N,32 yoki 16)."""\n    return _hash_core(msgs,keys,c,bits=msgs.shape[1]*8)\ndef _hash_core(msgs,keys,c,bits):\n    N,L=msgs.shape; bl=32 if c.p==256 else 16\n    nb=L//bl\n    blocks=[msgs[:,i*bl:(i+1)*bl] for i in range(nb)]\n    ny=[0]*N\n    nyb=np.zeros((N,bl),dtype=np.int64)\n    for n in range(N):\n        v=sum(int.from_bytes(bytes(b[n].tolist()),\'big\') for b in blocks)%(2**(8*bl))\n        nyb[n]=list(v.to_bytes(bl,\'big\'))\n    lb=np.tile(np.array(list(bits.to_bytes(bl,\'big\'))),(N,1))\n    kir=[to_units(b,c.p) for b in blocks]+[to_units(lb,c.p),to_units(nyb,c.p)]\n    k=to_units(keys,c.p); ke=k.copy(); holat=kir[0]\n    for i in range(1,len(kir)): holat,ke=iteration(holat,kir[i],ke,k,c)\n    return holat\n\n\n# ------------------------------------------------------------------ nazorat qiymatlari\nKAT_KALIT = bytes.fromhex("c86ed4b2756df1cd6b6eca81d441809221ab81bba0c7c15044be8feb934f95a2")\nKAT_XABAR = b"funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska"\nKAT = {\n    (256, \'std\'): "55394f2d84fccd4d8d01d8fef756cd657ebecc3f96f81e5207767447de66e2b6",\n    (256, \'sbz\'): "42f97daaa8e7010bc629ca695eaa77df7ae7443a0c3ae78d205c2e025d256c96",\n    (16, \'std\'):  "e9eaccdd4ccce2d229228c864989eb9f",\n    (16, \'sbz\'):  "bd745e2ed572ea4246ed68482ef46c5e",\n}\n\ndef xesh(xabar: bytes, kalit: bytes, p=256, bz=\'sbz\') -> bytes:\n    """Bitta xabar uchun xesh qiymat (xabar uzunligi blok uzunligiga karrali bo‘lishi shart emas)."""\n    bl = 32 if p == 256 else 16\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    # uzunlik bloki asl xabar uzunligi bo‘yicha hisoblanadi\n    c = Cfg(p, bz)\n    msgs = np.array([list(m)]); keys = np.array([list(kalit[:bl])])\n    out = _hash_core(msgs, keys, c, bits=len(xabar) * 8)[0]\n    if p == 16: out = [out[2*i] << 4 | out[2*i+1] for i in range(16)]\n    return bytes(int(x) for x in out)\n\ndef nazorat():\n    """A ilova va SBZ materialidagi nazorat qiymatlarini tekshiradi."""\n    ok = True\n    for (p, bz), kut in KAT.items():\n        r = xesh(KAT_XABAR, KAT_KALIT, p, bz).hex()\n        print(f"  p={p:3d}, {bz}: {\'MOS\' if r == kut else \'MOS EMAS\'}")\n        ok &= (r == kut)\n    return ok\n\nif __name__ == "__main__":\n    print("Nazorat qiymatlari:"); print("Hammasi mos" if nazorat() else "XATO")\n',
    'komponentlar': '"""Kriptotahlil uchun umumiy yordamchi funksiyalar."""\nimport itertools\nimport numpy as np\nfrom algoritm import Cfg, aralash, surholat\n\nPAR8 = np.array([bin(i).count(\'1\') & 1 for i in range(256)])\nX256 = np.arange(256)\n\ndef kalitlar(rng, N, p):\n    """Tasodifiy kalitlar: R (1-element) va 2-element juft (5.1.2-band talabi)."""\n    if p == 256:\n        k = rng.integers(0, 256, (N, 32)); k[:, 0] &= 0xFE; k[:, 1] &= 0xFE\n    else:\n        k = rng.integers(0, 16, (N, 32)); k[:, 0] &= 0xE; k[:, 1] &= 0xE\n    return k\n\ndef wht(f):\n    """Walsh–Adamar o‘zgartirishi (f — ±1 vektor)."""\n    a = f.astype(np.int64).copy(); h = 1; n = len(a)\n    while h < n:\n        a = a.reshape(-1, 2, h); a = np.stack([a[:, 0] + a[:, 1], a[:, 0] - a[:, 1]], 1).reshape(-1); h *= 2\n    return a\n\ndef lat(F, nout):\n    """8 bitli kirishli F uchun W[β][α] = Σ (−1)^(β·F(x) ⊕ α·x); korrelyatsiya c = W/256."""\n    W = np.zeros((1 << nout, 256), dtype=np.int64)\n    for b in range(1 << nout):\n        W[b] = wht(1 - 2 * PAR8[F & b])\n    return W\n\ndef ddt(F, nout):\n    """Ayirmalar matritsasi DDT[Δx][Δy] (8 bitli kirish)."""\n    return np.array([np.bincount(F ^ F[X256 ^ d], minlength=nout) for d in range(256)])\n\ndef anf_deg(fbits, nv):\n    """Boolean funksiyaning ANF darajasi (Möbius o‘zgartirishi)."""\n    a = fbits.astype(np.uint8).copy(); h = 1; N = 1 << nv\n    while h < N:\n        a = a.reshape(-1, 2, h); a[:, 1] ^= a[:, 0]; a = a.reshape(-1); h *= 2\n    wt = np.array([bin(i).count(\'1\') for i in range(N)]); nz = wt[a == 1]\n    return int(nz.max()) if len(nz) else -1\n\ndef bits(v, n): return (v[:, None] >> np.arange(n)) & 1\n\ndef implicit_count(points, deg):\n    """deg darajali chiziqli bog‘liq bo‘lmagan implitsit tenglamalar soni (monomlar matritsasi rangi orqali)."""\n    npts, nv = points.shape\n    mons = [()] + [m for d in range(1, deg + 1) for m in itertools.combinations(range(nv), d)]\n    words = (npts + 63) // 64; cols = []\n    for m in mons:\n        v = np.ones(npts, dtype=np.uint8)\n        for i in m: v &= points[:, i].astype(np.uint8)\n        pad = np.zeros(words * 64, dtype=np.uint8); pad[:npts] = v\n        cols.append(np.packbits(pad).view(np.uint64))\n    A = np.array(cols); rank = 0; n = A.shape[0]\n    for w in range(A.shape[1]):\n        for b in range(64):\n            if rank >= n: break\n            mask = np.uint64(1) << np.uint64(b)\n            idx = np.nonzero((A[rank:, w] & mask) != 0)[0]\n            if len(idx) == 0: continue\n            piv = rank + idx[0]\n            if piv != rank: A[[rank, piv]] = A[[piv, rank]]\n            rows = np.nonzero((A[:, w] & mask) != 0)[0]; rows = rows[rows != rank]\n            A[rows] ^= A[rank]; rank += 1\n    return len(mons) - rank, len(mons)\n\ndef qosh_elem(A1, B1, A2, B2, R, p=256):\n    """Qo‘sh o‘zgartirishining bitta elementi (parametrlar belgilangan): x → x^(−3)."""\n    x = np.arange(p); INV = np.zeros(p, dtype=np.int64)\n    for t in range(1, p, 2): INV[t] = pow(t, -1, p)\n    s = lambda x, A, B: (-A * x * INV[(B + R * x) % p]) % p\n    y1 = s(x, A1, B1); y2 = s(y1, A2, B2); y2 = np.where(y2 == 0, (A2 + B2) % p, y2)\n    return s(y2, A2, B2)\n\ndef tasodifiy_param(rng):\n    A1, B1, A2, B2 = [int(rng.integers(0, 128)) * 2 + 1 for _ in range(4)]\n    return A1, B1, A2, B2, int(rng.integers(0, 128)) * 2\n\ndef rank2(A):\n    A = A.copy() % 2; r = 0\n    for col in range(A.shape[1]):\n        piv = [i for i in range(r, A.shape[0]) if A[i, col]]\n        if not piv: continue\n        A[[r, piv[0]]] = A[[piv[0], r]]\n        for i in range(A.shape[0]):\n            if i != r and A[i, col]: A[i] ^= A[r]\n        r += 1\n    return r\n\n# 0-qatlam matritsalari: SurHolat (P) va Aralash (M(k0))\n_C2 = Cfg(2, \'std\')\n_perm = surholat(np.arange(32)[None, :].copy())[0]\nP_SUR = np.zeros((32, 32), dtype=np.int64); P_SUR[np.arange(32), _perm] = 1\n\ndef M_aralash(k):\n    """h0\' = M(k0)·h0 (mod 2) matritsasi."""\n    E = np.eye(32, dtype=np.int64); K = np.tile(np.asarray(k) % 2, (32, 1))\n    return (aralash(E, K, _C2).T) % 2\n\ndef L_raund(k):\n    """Yutishsiz raundning 0-qatlam matritsasi L = P·M(k0)."""\n    return (P_SUR @ M_aralash(k)) % 2\n',
    'differensial': '"""\n2-bo‘lim. Differensial kriptotahlil.\n  1) ayirmalar matritsalari: S, Z, SBZ, Qo‘sh elementi;\n  2) orttirmalarni raundlar va har bir akslantirish bo‘yicha kuzatish:\n       D0a/D0b — yutishdan keyingi holatdagi 1 / 2 baytli MSB ayirmasi,\n       D1 — iteratsiya boshidagi holatdagi 1 baytli MSB ayirmasi,\n       D2 — kirish blokidagi 2 baytli MSB ayirmasi, D3 — kirish blokidagi 1 baytli MSB ayirmasi;\n  3) Aralash yadrosi: nol va teng ustunli kalitlar ulushi.\nNatija: natijalar/differensial.json\n"""\nimport argparse, collections, json, os\nimport numpy as np\nfrom algoritm import S, Z, Cfg, iteration, aralash, qosh, surholat, surkalit, tuz, bzf\nfrom komponentlar import ddt, qosh_elem, tasodifiy_param, kalitlar, M_aralash\n\ndef komponentlar_tahlili(rng, trials=200):\n    R = {}; DS = ddt(S, 256); DZ = ddt(Z, 16); x = np.arange(256)\n    dist = lambda D: {int(a): int(b) for a, b in zip(*np.unique(D[1:], return_counts=True))}\n    R[\'S_max\'] = int(DS[1:].max()); R[\'S_taqsimot\'] = dist(DS)\n    R[\'S_bolagi\'] = [[int(DS[d, y]) for y in range(8)] for d in range(1, 9)]\n    R[\'Z_max\'] = int(DZ[1:].max()); i = np.unravel_index(DZ[1:].argmax(), DZ[1:].shape)\n    R[\'Z_max_joyi\'] = (hex(int(i[0]) + 1), hex(int(i[1]))); R[\'Z_taqsimot\'] = dist(DZ)\n    R[\'Z_sonishi_mumkin_ayirmalar\'] = int((DZ[1:, 0] > 0).sum())\n    for d in (0x80, 0x01, 0x96): R[f\'Z_satr_{d:02x}\'] = DZ[d].tolist()\n    def sbz(dh, dn):\n        if dn == 0: return DS[dh] / 256.0\n        return sum(DS[dn, d] * DS[dh ^ d] for d in range(256) if DS[dn, d]) / 65536.0\n    R[\'SBZ_dh0_dn80_max\'] = float(sbz(0, 0x80).max()); R[\'SBZ_dh80_dn0_max\'] = float(sbz(0x80, 0).max())\n    R[\'SBZ_dh0_dn80_sonish\'] = float(sbz(0, 0x80)[0]); R[\'SBZ_dh80_dn80_sonish\'] = float(sbz(0x80, 0x80)[0])\n    R[\'SBZ_80dan80ga\'] = float((DS[0x80] * DS[:, 0x80]).sum() / 65536)\n    mx, p80, pj = [], [], {j: [] for j in range(8)}\n    for _ in range(trials):\n        f = qosh_elem(*tasodifiy_param(rng)); D = ddt(f, 256)\n        mx.append(int(D[1:].max())); p80.append(D[0x80, 0x80] / 256)\n        for j in range(8): pj[j].append(D[1 << j].max() / 256)\n    R[\'Qosh_max\'] = int(max(mx)); R[\'Qosh_p(80->80)\'] = float(np.mean(p80))\n    R[\'Qosh_2^j_eng_ehtimolli\'] = [round(float(np.mean(pj[j])), 4) for j in range(8)]\n    return R\n\ndef raundlar_yutishsiz(holat, blok, ke, k, c, trace):\n    """Yutishdan keyingi holatdan boshlab 1–10-raundlar (0-nuqtadagi yutishsiz)."""\n    R = k[:, 0]\n    def T(nm, j): trace.append((nm, j, holat.copy(), ke.copy()))\n    for j in range(1, 11):\n        holat = aralash(holat, ke, c); T(\'Aralash\', j)\n        holat, ke = qosh(holat, ke, R, c); T(\'Qosh1\', j)\n        holat = surholat(holat); ke = surkalit(ke); T(\'Sur\', j)\n        if j == 8:\n            holat, hn = qosh(holat, blok, R, c); T(\'Qosh_inj8\', j); holat = bzf(holat, hn, c); T(\'BZ8\', j)\n        else:\n            holat, ke = qosh(holat, ke, R, c); T(\'Qosh2\', j)\n        ke = tuz(ke, k, c); T(\'Tuz\', j)\n        if j == 9:\n            holat, hn = qosh(holat, blok, R, c); T(\'Qosh_inj9\', j); holat = bzf(holat, hn, c); T(\'BZ9\', j)\n\ndef hw(a, p): return sum(((a >> i) & 1) for i in range(8 if p == 256 else 4)).sum(1)\n\ndef kuzatish(rng, N=2000, holatlar=(\'D1\', \'D2\', \'D3\')):\n    R = {}\n    for p, bz in [(256, \'std\'), (256, \'sbz\'), (16, \'std\'), (16, \'sbz\')]:\n        c = Cfg(p, bz); msb = p // 2\n        for case in holatlar:\n            k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32))\n            h2, bl2 = h.copy(), bl.copy()\n            if case in (\'D0a\', \'D1\'): h2[:, 0] ^= msb\n            elif case == \'D0b\': h2[:, 0] ^= msb; h2[:, 9] ^= msb\n            elif case == \'D2\': bl2[:, 0] ^= msb; bl2[:, 9] ^= msb\n            else: bl2[:, 0] ^= msb\n            t1, t2 = [], []\n            if case.startswith(\'D0\'):\n                raundlar_yutishsiz(h, bl, k.copy(), k, c, t1); raundlar_yutishsiz(h2, bl, k.copy(), k, c, t2)\n            else:\n                iteration(h, bl, k.copy(), k, c, trace=t1); iteration(h2, bl2, k.copy(), k, c, trace=t2)\n            rows = []\n            for (nm, j, a, ka), (_, _, b, kb) in zip(t1, t2):\n                d, dk = a ^ b, ka ^ kb; act = (d != 0).sum(1)\n                msbonly = (((d == 0) | (d == msb)).all(1) & ((dk == 0) | (dk == msb)).all(1)).mean()\n                top = collections.Counter(map(bytes, np.concatenate([d, dk], 1).astype(np.uint8))).most_common(1)[0][1] / N\n                rows.append(dict(amal=nm, raund=j, faol_holat=round(float(act.mean()), 2), faol_holat_min=int(act.min()),\n                                 faol_ke=round(float((dk != 0).sum(1).mean()), 2), P_MSB_izi=round(float(msbonly), 4),\n                                 eng_ehtimolli_ayirma=round(top, 4), xemming=round(float(hw(d, p).mean()), 1),\n                                 nol_ayirma=round(float(((act == 0) & ((dk != 0).sum(1) == 0)).mean()), 4)))\n            R[f\'{p}_{bz}_{case}\'] = rows\n        print(f"  p={p}, {bz}: tayyor")\n    return R\n\ndef yadro(rng, nkeys=300):\n    zc = eq = 0\n    for _ in range(nkeys):\n        M = M_aralash(kalitlar(rng, 1, 256)[0]); cols = [tuple(M[:, i]) for i in range(32)]\n        zc += any(sum(cc) == 0 for cc in cols); eq += len(set(cols)) < 32\n    return {\'nol_ustunli_kalitlar\': zc / nkeys, \'teng_ustunli_kalitlar\': eq / nkeys}\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--tez\', action=\'store_true\')\n    a = ap.parse_args(); s = 0.15 if a.tez else 1.0\n    os.makedirs(\'natijalar\', exist_ok=True); out = {}\n    print("1. Ayirmalar matritsalari..."); out[\'komponentlar\'] = komponentlar_tahlili(np.random.default_rng(11), int(200 * s))\n    print("2. Orttirmalarni kuzatish...")\n    out[\'kuzatish\'] = kuzatish(np.random.default_rng(5), int(2000 * s), (\'D1\', \'D2\', \'D3\'))\n    out[\'kuzatish\'].update(kuzatish(np.random.default_rng(9), int(3000 * s), (\'D0a\', \'D0b\')))\n    print("3. Aralash yadrosi..."); out[\'yadro\'] = yadro(np.random.default_rng(31), int(300 * s))\n    json.dump(out, open(\'natijalar/differensial.json\', \'w\'), indent=1, ensure_ascii=False)\n    print("Saqlandi: natijalar/differensial.json")\n',
    'variant': '"""\nTavsiyalar bo‘yicha o‘zgartirilgan 1-algoritm variantlari (vektorlashtirilgan).\n  bz    : \'std\' (BaytZichlash) yoki \'sbz\' (S(h ⊕ S(n)))      — 1-tavsiya\n  toq   : True — kₑ elementlari doim toq (kₑ ← kₑ ∨ 1)         — 2-tavsiya\n  sqatlam: True — raund ichida S-box qatlami (SurHolat dan keyin) — 3-tavsiya\n"""\nimport numpy as np\nfrom algoritm import Cfg, aralash, qosh, surholat, surkalit, tuz, bzf, S, to_units\n\nPRESENT = np.array([0xC, 5, 6, 0xB, 9, 0, 0xA, 0xD, 3, 0xE, 0xF, 8, 4, 7, 1, 2])\n\ndef s_qatlam(h, p):\n    return S[h] if p == 256 else PRESENT[h]\n\ndef iteration_v(holat, blok, ke, k, c, toq=False, sqatlam=False, trace=None, rounds=10):\n    R = k[:, 0]\n    def T(nm, j):\n        if trace is not None: trace.append((nm, j, holat.copy(), ke.copy()))\n    if toq: ke = ke | 1\n    holat, hn = qosh(holat, blok, R, c); T(\'Qosh0\', 0)\n    holat = bzf(holat, hn, c); T(\'BZ0\', 0)\n    for j in range(1, rounds + 1):\n        holat = aralash(holat, ke, c); T(\'Aralash\', j)\n        holat, ke = qosh(holat, ke, R, c); T(\'Qosh1\', j)\n        holat = surholat(holat); ke = surkalit(ke); T(\'Sur\', j)\n        if sqatlam: holat = s_qatlam(holat, c.p); T(\'Sqatlam\', j)\n        if j == 8:\n            holat, hn = qosh(holat, blok, R, c); holat = bzf(holat, hn, c); T(\'BZ8\', j)\n        else:\n            holat, ke = qosh(holat, ke, R, c); T(\'Qosh2\', j)\n        ke = (ke | 1) if toq else tuz(ke, k, c); T(\'Tuz\', j)\n        if j == 9:\n            holat, hn = qosh(holat, blok, R, c); holat = bzf(holat, hn, c); T(\'BZ9\', j)\n    return holat, ke\n\ndef xesh_v(xabar: bytes, kalit: bytes, p=256, bz=\'sbz\', toq=False, sqatlam=False) -> bytes:\n    c = Cfg(p, bz); bl = 32 if p == 256 else 16\n    bits = len(xabar) * 8\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    bloks = [np.array(list(m[i:i + bl])) for i in range(0, len(m), bl)]\n    ny = sum(int.from_bytes(bytes(b.tolist()), \'big\') for b in bloks) % 2 ** (8 * bl)\n    kir = [to_units(b, p) for b in bloks] + [to_units(np.array(list(bits.to_bytes(bl, \'big\'))), p),\n                                             to_units(np.array(list(ny.to_bytes(bl, \'big\'))), p)]\n    k = to_units(np.array(list(kalit[:bl])), p)[None]; ke = k.copy(); h = kir[0][None]\n    for i in range(1, len(kir)):\n        h, ke = iteration_v(h, kir[i][None], ke, k, c, toq, sqatlam)\n    h = h[0]\n    if p == 16: h = [h[2 * i] << 4 | h[2 * i + 1] for i in range(16)]\n    return bytes(int(v) for v in h)\n',
}
for _nom, _matn in _MODULLAR.items():
    if importlib.util.find_spec(_nom) is not None and _nom not in sys.modules:
        print(f"{_nom}.py — papkadagi fayldan olinadi"); continue
    _m = types.ModuleType(_nom); _m.__file__ = f"<{_nom}>"
    sys.modules[_nom] = _m; exec(compile(_matn, f"<{_nom}>", "exec"), _m.__dict__)
    print(f"{_nom} — daftar ichidan yuklandi")


algoritm — daftar ichidan yuklandi
komponentlar — daftar ichidan yuklandi
differensial — daftar ichidan yuklandi
variant — daftar ichidan yuklandi


In [2]:
import numpy as np, itertools, json, time
from algoritm import *
from komponentlar import *
from variant import xesh_v, iteration_v, PRESENT
TEZ = False                      # True — tez sinov (kichik tanlanmalar)
K_ = (lambda n: max(1, n // 5)) if TEZ else (lambda n: n)
R = {}
print("Nazorat qiymatlari (standart va SBZ):", "hammasi mos" if nazorat() else "XATO")
# Tavsiyalar bo‘yicha variantlarning nazorat qiymatlari (A ilova kaliti va xabari)
VAR = [('std', False, False), ('sbz', False, False), ('std', True, False), ('std', False, True), ('sbz', True, False), ('sbz', True, True)]
R['KAT'] = {}
for p in (256, 16):
    for bz, toq, sq in VAR:
        h = xesh_v(KAT_XABAR, KAT_KALIT, p, bz, toq, sq).hex()
        R['KAT'][f"{p}_{bz}_{int(toq)}_{int(sq)}"] = h
        print(f"p={p:3d}  bz={bz}  toq kₑ={int(toq)}  S-qatlam={int(sq)}:  {h}")

  p=256, std: MOS
  p=256, sbz: MOS
  p= 16, std: MOS
  p= 16, sbz: MOS
Nazorat qiymatlari (standart va SBZ): hammasi mos
p=256  bz=std  toq kₑ=0  S-qatlam=0:  55394f2d84fccd4d8d01d8fef756cd657ebecc3f96f81e5207767447de66e2b6
p=256  bz=sbz  toq kₑ=0  S-qatlam=0:  42f97daaa8e7010bc629ca695eaa77df7ae7443a0c3ae78d205c2e025d256c96
p=256  bz=std  toq kₑ=1  S-qatlam=0:  073956b2e84a40721e49415fca606b118fc65c174d511fda3fde998f75a024fb
p=256  bz=std  toq kₑ=0  S-qatlam=1:  9f9c061f419d14358fe7c935dc0afc77561dc61e0e2d23c4aa32c6a7ff044e7c
p=256  bz=sbz  toq kₑ=1  S-qatlam=0:  2e632c6921f4ecf801e1a5ab1d056a5b1425ee6117a17d4ec8b3e982d66b10b7
p=256  bz=sbz  toq kₑ=1  S-qatlam=1:  240aa5e0f429d55372144ea1091c7c50caea0c623c659b34cf82f1e04962c4ba
p= 16  bz=std  toq kₑ=0  S-qatlam=0:  e9eaccdd4ccce2d229228c864989eb9f
p= 16  bz=sbz  toq kₑ=0  S-qatlam=0:  bd745e2ed572ea4246ed68482ef46c5e
p= 16  bz=std  toq kₑ=1  S-qatlam=0:  c548c7a15798b147b51ea774d174b6c1
p= 16  bz=std  toq kₑ=0  S-qatlam=1:  9bf627ae9

## 1. 2-tavsiya: kₑ ning juft/toqlik naqshi

Aralash 2 moduli bo‘yicha faqat kₑ elementlarining juft/toqligiga bog‘liq chiziqli akslantirish M(k₀).
Har bir 4×4 yarim uchun barcha 2¹⁶ naqsh tekshiriladi: qaysilarida yarim matritsa teskarilanuvchi va
ularning tarmoqlanish soni (min wt(x) + wt(Mx)) qancha.

In [3]:
from algoritm import dia
def M_yarim(bitlar):
    K = np.array(bitlar, dtype=np.int64).reshape(4, 4)
    E = np.eye(16, dtype=np.int64).reshape(16, 4, 4)
    return (dia(E, np.tile(K, (16, 1, 1)), 2).reshape(16, 16).T) % 2
asos = [M_yarim([1 if i == b else 0 for i in range(16)]) for b in range(16)]   # M(K) K ga nisbatan chiziqli
def M_of(p_):
    M = np.zeros((16, 16), dtype=np.int64)
    for b in range(16):
        if (p_ >> b) & 1: M ^= asos[b]
    return M
w2 = 1 << np.arange(16)
def rang_bit(rows):
    rows = list(rows); r = 0
    for b in range(16):
        piv = next((i for i in range(r, len(rows)) if (rows[i] >> b) & 1), None)
        if piv is None: continue
        rows[r], rows[piv] = rows[piv], rows[r]
        for i in range(len(rows)):
            if i != r and (rows[i] >> b) & 1: rows[i] ^= rows[r]
        r += 1
    return r
invs = [p_ for p_ in range(65536) if rang_bit((M_of(p_) * w2).sum(1)) == 16]
pc = np.array([bin(i).count('1') for i in range(65536)])
def tarmoqlanish(p_):
    cols = (M_of(p_) * w2[:, None]).sum(0); lo = np.zeros(256, dtype=np.int64); hi = np.zeros(256, dtype=np.int64)
    for x in range(1, 256):
        b = (x & -x).bit_length() - 1; lo[x] = lo[x & (x - 1)] ^ cols[b]; hi[x] = hi[x & (x - 1)] ^ cols[b + 8]
    img = (hi[:, None] ^ lo[None, :]).reshape(-1); return int((pc + pc[img])[1:].min())
br = {p_: tarmoqlanish(p_) for p_ in invs}
from collections import Counter
taq = dict(sorted(Counter(br.values()).items())); eng = [p_ for p_, b in br.items() if b == max(taq)]
print(f"Teskarilanuvchi naqshlar: {len(invs)} / 65536 = {len(invs)/65536:.3f}")
print("Tarmoqlanish soni bo‘yicha taqsimot:", taq)
print("Eng katta tarmoqlanishli naqsh(lar):", [hex(p_) for p_ in eng], "— 0xffff = barcha elementlar toq")
R['naqsh'] = dict(teskarilanuvchi=len(invs), taqsimot={str(k): v for k, v in taq.items()}, eng=[hex(p_) for p_ in eng])

Teskarilanuvchi naqshlar: 7568 / 65536 = 0.115
Tarmoqlanish soni bo‘yicha taqsimot: {2: 5532, 3: 1955, 4: 80, 6: 1}
Eng katta tarmoqlanishli naqsh(lar): ['0xffff'] — 0xffff = barcha elementlar toq


### 1.1. Toq kₑ bilan L = P·M matritsasi va faol baytlar zanjiri

In [4]:
Lf = (P_SUR @ M_aralash(np.ones(32, dtype=np.int64))) % 2
print(f"rang L = {rank2(Lf)},  nol satrlar = {int((Lf.sum(1)==0).sum())},  nol ustunlar = {int((Lf.sum(0)==0).sum())}")
def zanjir(Lm, wmax=2):
    L8 = np.linalg.matrix_power(Lm, 8) % 2; maps = [L8, Lm, Lm, L8, Lm]; best = {k: 99 for k in range(1, 7)}
    cands = [np.eye(32, dtype=np.int64)[list(c)].sum(0) % 2 for w in range(1, wmax + 1) for c in itertools.combinations(range(32), w)]
    for end in range(1, 6):
        for u in cands:
            tot = int(u.sum()); cur = u; n = 1; best[1] = min(best[1], tot)
            for m in range(end - 1, -1, -1):
                cur = (maps[m].T @ cur) % 2
                if cur.sum() == 0: break
                tot += int(cur.sum()); n += 1; best[n] = min(best[n], tot)
    return best
zf = zanjir(Lf)
rng = np.random.default_rng(9); zs = {k: [] for k in range(1, 7)}
for t in range(K_(60)):
    zz = zanjir(L_raund(kalitlar(rng, 1, 256)[0]))
    for k_ in zz:
        if zz[k_] < 99: zs[k_].append(zz[k_])
print("Yutish qatlamlari:      ", list(range(1, 7)))
print("Hozirgi: minimum        ", [min(zs[k_]) for k_ in range(1, 7)])
print("Hozirgi: mediana        ", [float(np.median(zs[k_])) for k_ in range(1, 7)])
print("Toq kₑ (har qanday kalit)", [zf[k_] for k_ in range(1, 7)])
R['L_toq'] = dict(rang=rank2(Lf), zanjir_toq=[zf[k_] for k_ in range(1, 7)],
                  zanjir_hozirgi_min=[min(zs[k_]) for k_ in range(1, 7)], zanjir_hozirgi_med=[float(np.median(zs[k_])) for k_ in range(1, 7)])

rang L = 32,  nol satrlar = 0,  nol ustunlar = 0
Yutish qatlamlari:       [1, 2, 3, 4, 5, 6]
Hozirgi: minimum         [1, 2, 3, 4, 5, 6]
Hozirgi: mediana         [1.0, 2.0, 4.0, 15.0, 31.5, 43.0]
Toq kₑ (har qanday kalit) [1, 4, 9, 24, 43, 57]


### 1.2. Chiqishdagi munosabatlar, doimiy bitlar va ayirmalarning so‘nishi

Hozirgi va toq kₑ li algoritm solishtiriladi: (a) 1000 ta xesh qiymatning 0-qatlami vektorlari fazosining rangi
(32 bo‘lsa, doimiy munosabatlar yo‘q) va har bir bitning chastotasi; (b) yutishdan keyingi holatdagi bitta MSB
ayirmasining raundlarda butunlay so‘nish ulushi va MSB izining saqlanishi.

In [5]:
def raundlar(h, bl, ke, k, c, toq=False, E=None, nr=10, trace=None):
    """0-nuqtadagi yutishsiz 1..nr raundlar; E — raund ichidagi qo‘shimcha element (SurHolat dan keyin)."""
    Rr = k[:, 0]
    if toq: ke = ke | 1
    for j in range(1, nr + 1):
        h = aralash(h, ke, c); h, ke = qosh(h, ke, Rr, c); h = surholat(h); ke = surkalit(ke)
        if E is not None: h = E(h)
        if j == 8: h, hn = qosh(h, bl, Rr, c); h = bzf(h, hn, c)
        else: h, ke = qosh(h, ke, Rr, c)
        ke = (ke | 1) if toq else tuz(ke, k, c)
        if j == 9: h, hn = qosh(h, bl, Rr, c); h = bzf(h, hn, c)
        if trace is not None: trace.append((j, h.copy(), ke.copy()))
    return h, ke
R['chiqish'] = {}; R['sonish'] = {}
rng = np.random.default_rng(71)
for p in (256, 16):
    for toq in (False, True):
        ranks, fmin, fmax = [], 1, 0
        for t in range(3):
            if p == 256: k = kalitlar(rng, 1, 256)[0]
            else: k = rng.integers(0, 256, 16); k[0] &= 0xEE
            m = rng.integers(0, 256, (K_(1000), 64)); o = np.array([list(xesh_v(bytes(mm.tolist()), bytes(k.tolist()), p, 'std', toq, False)) for mm in m[:K_(400)]])
            ob = (to_units(o, p) % 2); ranks.append(rank2((ob[1:] ^ ob[0]).T.copy())); f = ob.mean(0); fmin = min(fmin, f.min()); fmax = max(fmax, f.max())
        R['chiqish'][f'{p}_{int(toq)}'] = dict(rang=ranks, chastota=[round(float(fmin), 3), round(float(fmax), 3)])
        print(f"p={p:3d}, toq kₑ={int(toq)}: 0-qatlam rangi {ranks},  bit chastotalari {fmin:.3f}…{fmax:.3f}")
for p in (256, 16):
    for toq in (False, True):
        c = Cfg(p, 'std'); N = K_(3000); msb = p // 2; k = kalitlar(rng, N, p)
        h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32)); h2 = h.copy(); h2[:, 0] ^= msb
        t1, t2 = [], []; raundlar(h, bl, k.copy(), k, c, toq, trace=t1); raundlar(h2, bl, k.copy(), k, c, toq, trace=t2)
        z = [round(float(((a[1] == b[1]).all(1) & (a[2] == b[2]).all(1)).mean()), 4) for a, b in zip(t1, t2)]
        mi = [round(float((((a[1] ^ b[1]) == 0) | ((a[1] ^ b[1]) == msb)).all(1).mean()), 3) for a, b in zip(t1, t2)]
        R['sonish'][f'{p}_{int(toq)}'] = dict(nol=z, msb=mi)
        print(f"p={p:3d}, toq kₑ={int(toq)}: so‘nish ulushi {z[:3]} …;  MSB izi {mi[:7]}")

p=256, toq kₑ=0: 0-qatlam rangi [27, 24, 22],  bit chastotalari 0.000…0.542
p=256, toq kₑ=1: 0-qatlam rangi [32, 32, 32],  bit chastotalari 0.440…0.585
p= 16, toq kₑ=0: 0-qatlam rangi [30, 23, 25],  bit chastotalari 0.000…0.565
p= 16, toq kₑ=1: 0-qatlam rangi [32, 32, 32],  bit chastotalari 0.453…0.562
p=256, toq kₑ=0: so‘nish ulushi [0.0307, 0.0327, 0.033] …;  MSB izi [0.943, 0.827, 0.711, 0.611, 0.504, 0.406, 0.34]
p=256, toq kₑ=1: so‘nish ulushi [0.0, 0.0, 0.0] …;  MSB izi [0.954, 0.829, 0.733, 0.652, 0.571, 0.451, 0.384]
p= 16, toq kₑ=0: so‘nish ulushi [0.0263, 0.028, 0.028] …;  MSB izi [0.47, 0.164, 0.064, 0.037, 0.029, 0.029, 0.028]
p= 16, toq kₑ=1: so‘nish ulushi [0.0, 0.0, 0.0] …;  MSB izi [0.502, 0.077, 0.011, 0.001, 0.0, 0.0, 0.0]


## 2. 3-tavsiya: raund ichidagi T-funksiyani buzuvchi element

Uchta element SurHolat dan keyin holatning har bir elementiga qo‘llanadi va hozirgi algoritm bilan solishtiriladi:
- 0-qatlam raund tenglamasi h₀ ← L·h₀ ning korrelyatsiyasi (raundlar bo‘yicha);
- 1 raund: eng yaxshi «bitta bit → bitta bit» korrelyatsiyasi (shovqin chegarasi bilan);
- MSB izining saqlanishi;
- ko‘chki (SAC): 1 va 2 raunddan keyin o‘rtacha qiymat va bog‘liqligi yo‘q (kirish, chiqish) bit juftliklari ulushi;
- MSB ning 0-bitga ta’siri (1 va 2 raund);
- integral: bitta faol bayt bo‘yicha 0-qatlam yig‘indisi nol bo‘lish ulushi (1–3-raund);
- hisoblash vaqti.

In [6]:
def element(nom, p):
    if nom == 'yoq': return None
    if p == 256:
        return {'aylantirish': lambda h: ((h << 3) | (h >> 5)) & 255, 'xor_surish': lambda h: h ^ (h >> 4), 'sbox': lambda h: S[h]}[nom]
    return {'aylantirish': lambda h: ((h << 1) | (h >> 3)) & 15, 'xor_surish': lambda h: h ^ (h >> 2), 'sbox': lambda h: PRESENT[h]}[nom]
def bitlar(a, p):
    nb = 8 if p == 256 else 4
    return ((a[:, :, None] >> np.arange(nb)) & 1).reshape(len(a), -1)
R['element'] = {}
rng = np.random.default_rng(81)
for p in (256, 16):
    c = Cfg(p, 'std'); msb = p // 2; nb = 8 if p == 256 else 4
    for nom in ('yoq', 'aylantirish', 'xor_surish', 'sbox'):
        E = element(nom, p); r = {}
        N = K_(600); k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32)); tr = []
        raundlar(h, bl, k.copy(), k, c, False, E, trace=tr)
        Ls = [L_raund(k[n]) for n in range(N)]; cur = h % 2; cl = []
        for j, hh, kk in tr:
            cur = np.array([(Ls[n] @ cur[n]) % 2 for n in range(N)]); cl.append(round(2 * float((cur == hh % 2).mean()) - 1, 3))
        r['qatlam0_c'] = cl
        N = K_(6000); k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32))
        o, _ = raundlar(h, bl, k.copy(), k, c, False, E, nr=1)
        X = 1 - 2 * bitlar(h, p).astype(float); Y = 1 - 2 * bitlar(o, p).astype(float)
        r['bit_bit'] = round(float((np.abs(Y.T @ X) / N).max()), 3); r['shovqin'] = round(4.5 / np.sqrt(N), 3)
        N = K_(2000); k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32)); h2 = h.copy(); h2[:, 0] ^= msb
        t1, t2 = [], []; raundlar(h, bl, k.copy(), k, c, False, E, trace=t1); raundlar(h2, bl, k.copy(), k, c, False, E, trace=t2)
        r['msb_izi'] = [round(float(((((a[1] ^ b[1]) == 0) | ((a[1] ^ b[1]) == msb)).all(1) & (((a[2] ^ b[2]) == 0) | ((a[2] ^ b[2]) == msb)).all(1)).mean()), 3) for a, b in zip(t1, t2)]
        for nr in (1, 2):
            N = K_(300); k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32))
            o, _ = raundlar(h, bl, k.copy(), k, c, False, E, nr=nr); ob = bitlar(o, p); SAC = np.zeros((32 * nb, 32 * nb))
            for i in range(32 * nb):
                h2 = h.copy(); h2[:, i // nb] ^= (1 << (i % nb)); o2, _ = raundlar(h2, bl, k.copy(), k, c, False, E, nr=nr)
                SAC[i] = (ob ^ bitlar(o2, p)).mean(0)
            hi = [i for i in range(32 * nb) if i % nb == nb - 1]; lo = [i for i in range(32 * nb) if i % nb == 0]
            r[f'sac{nr}'] = dict(ortacha=round(float(SAC.mean()), 3), nol=round(float((SAC == 0).mean()), 3), min=round(float(SAC.min()), 3))
            r[f'msb_0bit{nr}'] = round(float(SAC[np.ix_(hi, lo)].mean()), 3)
        zf_ = []
        for t in range(K_(60)):
            k = kalitlar(rng, 1, p)[0]; h = rng.integers(0, p, 32); bl = rng.integers(0, p, 32)
            H = np.tile(h, (p, 1)); H[:, 0] = np.arange(p); Kk = np.tile(k, (p, 1)); tr = []
            raundlar(H, np.tile(bl, (p, 1)), Kk.copy(), Kk, c, False, E, nr=3, trace=tr)
            zf_.append([float(((np.bitwise_xor.reduce(hh, axis=0) & 1) == 0).mean()) for j, hh, kk in tr])
        r['integral0'] = [round(float(v), 2) for v in np.mean(zf_, 0)]
        N = K_(2000); k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32))
        t0 = time.perf_counter(); raundlar(h, bl, k.copy(), k, c, False, E); r['vaqt'] = round(time.perf_counter() - t0, 3)
        R['element'][f'{p}_{nom}'] = r
        print(f"p={p:3d} {nom:12s} 0-qatlam c: {cl[:3]}  bit→bit: {r['bit_bit']} (shovqin {r['shovqin']})  MSB izi: {r['msb_izi'][:2]}"
              f"  SAC1: {r['sac1']}  SAC2 nol: {r['sac2']['nol']}  MSB→0: {r['msb_0bit1']}/{r['msb_0bit2']}  integral: {r['integral0']}  vaqt: {r['vaqt']}")

p=256 yoq          0-qatlam c: [1.0, 1.0, 1.0]  bit→bit: 0.142 (shovqin 0.058)  MSB izi: [0.944, 0.835]  SAC1: {'ortacha': 0.214, 'nol': 0.124, 'min': 0.0}  SAC2 nol: 0.115  MSB→0: 0.0/0.0  integral: [1.0, 1.0, 1.0]  vaqt: 0.275
p=256 aylantirish  0-qatlam c: [-0.011, 0.007, -0.0]  bit→bit: 0.057 (shovqin 0.058)  MSB izi: [0.031, 0.031]  SAC1: {'ortacha': 0.45, 'nol': 0.0, 'min': 0.0}  SAC2 nol: 0.0  MSB→0: 0.014/0.481  integral: [0.78, 0.52, 0.51]  vaqt: 0.27
p=256 xor_surish   0-qatlam c: [0.002, -0.003, 0.007]  bit→bit: 0.056 (shovqin 0.058)  MSB izi: [0.03, 0.03]  SAC1: {'ortacha': 0.426, 'nol': 0.002, 'min': 0.0}  SAC2 nol: 0.0  MSB→0: 0.007/0.466  integral: [0.82, 0.53, 0.53]  vaqt: 0.292
p=256 sbox         0-qatlam c: [0.086, 0.024, -0.006]  bit→bit: 0.058 (shovqin 0.058)  MSB izi: [0.035, 0.035]  SAC1: {'ortacha': 0.468, 'nol': 0.0, 'min': 0.063}  SAC2 nol: 0.0  MSB→0: 0.139/0.48  integral: [0.69, 0.52, 0.53]  vaqt: 0.282
p= 16 yoq          0-qatlam c: [1.0, 1.0, 1.0]  bit→bit:

## 3. Uchala tavsiya birgalikda (SBZ + toq kₑ + S-box qatlami)

In [7]:
rng = np.random.default_rng(9); R['birlashgan'] = {}
for p in (256, 16):
    c = Cfg(p, 'sbz'); N = K_(3000); msb = p // 2; k = kalitlar(rng, N, p)
    h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32)); h2 = h.copy(); h2[:, 0] ^= msb
    E = element('sbox', p); t1, t2 = [], []
    raundlar(h, bl, k.copy(), k, c, True, E, nr=7, trace=t1); raundlar(h2, bl, k.copy(), k, c, True, E, nr=7, trace=t2)
    mi = [round(float((((a[1] ^ b[1]) == 0) | ((a[1] ^ b[1]) == msb)).all(1).mean()), 4) for a, b in zip(t1, t2)]
    z = [round(float(((a[1] == b[1]).all(1) & (a[2] == b[2]).all(1)).mean()), 4) for a, b in zip(t1, t2)]
    R['birlashgan'][str(p)] = dict(msb=mi, nol=z)
    print(f"p={p}: MSB izi (1–7-raund) {mi};  ayirma so‘ngan juftliklar {z}")
json.dump(R, open('tavsiyalar_natijalari.json', 'w'), ensure_ascii=False, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else int(o))
print("Natijalar tavsiyalar_natijalari.json ga yozildi.")

p=256: MSB izi (1–7-raund) [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0];  ayirma so‘ngan juftliklar [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
p=16: MSB izi (1–7-raund) [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0];  ayirma so‘ngan juftliklar [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Natijalar tavsiyalar_natijalari.json ga yozildi.
